> ### **Salesperson Clean**

In [0]:
# Define paths for the Medallion Architecture
BRONZE_PATH = "/Volumes/workspace/accenture_final_project/lakehouse/bronze/"
SILVER_PATH = "/Volumes/workspace/accenture_final_project/lakehouse/silver/"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [0]:
# Define schema to enforce data types upon reading
salesperson_schema = StructType([
    StructField("EmployeeKey", IntegerType(), True),
    StructField("EmployeeID", IntegerType(), True),
    StructField("Salesperson", StringType(), True),
    StructField("Title", StringType(), True),
    StructField("UPN", StringType(), True)
])

In [0]:
# Read from Bronze layer
df_salesperson_raw = (spark.read
                 .format("csv")
                 .schema(salesperson_schema)
                 .option("header", "true")
                 .option("sep", "\t")                      
                 .option("ignoreLeadingWhiteSpace", "true")   
                 .option("ignoreTrailingWhiteSpace", "true")            
                 .option("quote", '"')
                 .option("escape", '"')
                 .load(BRONZE_PATH + "Salesperson.csv")       
                )

In [0]:
display(df_salesperson_raw)

In [0]:
# Count initial rows for Data Quality metrics
initial_count = df_salesperson_raw.count()

In [0]:
# Silver Layer Transformations
df_salesperson_silver = (df_salesperson_raw
    # Basic standardization of text fields: Trim whitespaces
    .withColumn("Salesperson", F.trim(F.col("Salesperson")))
    .withColumn("Title", F.trim(F.col("Title")))
    .withColumn("UPN", F.trim(F.col("UPN")))

    # Address duplicate records
    .dropDuplicates()
)

In [0]:
display(df_salesperson_silver)

In [0]:
# --- OUTLIER & ANOMALY HANDLING ---
# Filter out records with missing IDs or logically invalid negative IDs
df_salesperson_clean = df_salesperson_silver.filter(
    (F.col("EmployeeID").isNotNull()) & 
    (F.col("EmployeeKey").isNotNull()) &
    (F.col("EmployeeID") > 0)
)

In [0]:
# Calculate metrics for the presentation
final_count = df_salesperson_clean.count()
print(f"Data Quality Metrics for Salesperson:")
print(f"Initial rows (Bronze): {initial_count}")
print(f"Final valid rows (Silver): {final_count}")
print(f"Rows removed (Duplicates/Outliers/Nulls): {initial_count - final_count}")
print(f"Data Retention Rate: {(final_count / initial_count) * 100:.2f}%")

## **Transform to DELTA**

In [0]:
# Write to Silver Layer as Delta format
(df_salesperson_clean.write
 .mode("overwrite")
 .format("delta")
 .saveAsTable("accenture_final_project.salesperson_clean"))